# Polynomial coefficient views

Adapted from Qualtran v0.7.0 gf_poly_split_and_join.ipynb at bcb17bb25360ed68ce3bc084645db3e929799389. Copyright 2025 Google LLC, Apache-2.0. Both original degree-four GF(8) factories and directional register signatures are preserved. Native table layout and operation names are explicit presentation adaptations.


In [ ]:
from IPython.display import SVG, display
from examples.algorithms.field_polynomial_catalog import polynomial_view_examples
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.compiler.registers.finite_field.field_polynomial_layout import FieldPolynomialLayout
from flagquantum.simulation.finite_field_addition import binary_field_encoding
from flagquantum.simulation.finite_field_polynomial_encoding import FiniteFieldPolynomialEncoding
from examples.algorithms.binary_field_arithmetic import FieldOperation
import flagquantum as fq
import torch


## Split

A degree-n polynomial f(x) = sum a_i x^i over GF(p^m) occupies n+1 coefficient words. Splitting views the packed register as |a_n>|a_(n-1)>...|a_0>. Bits within each coefficient are big-endian. The polynomial dtype is input-only and the coefficient array is output-only. The view reuses caller-owned wires, with zero gates, work wires and measurements.


In [ ]:
examples = polynomial_view_examples()
split_example = examples[0]
split_example


### Graphical Signature


In [ ]:
display(SVG(field_signature_svg([(split_example[0], split_example[1], {})])))


## Join

Joining reverses the view and restores the packed polynomial. Its coefficient array is input-only and its polynomial register is output-only. Both examples use degree four and GF(8). FieldPolynomialLayout accepts concrete dimensions. The original validator rejects symbolic polynomial degree, while a symbolic field descriptor can remain lazy metadata. Native executable words require a concrete coefficient width.


In [ ]:
join_example = examples[1]
join_example


### Graphical Signature


In [ ]:
display(SVG(field_signature_svg([(join_example[0], join_example[1], {})])))


## Classical values, view inverse and append

Simulation owns padded polynomial coefficients and reconstruction. Compiler owns physical word grouping. This small degree-two GF(4) instance checks the original identity-tensor scenario using a coherent state and normal append. A noncontiguous caller binding is restored exactly. There is one coherent branch and no allocated workspace.


In [ ]:
encoding = FiniteFieldPolynomialEncoding(2, binary_field_encoding(2))
polynomial = encoding.from_gf_coefficients(encoding.field.gf_type([1, 2, 3]))
coefficients = encoding.to_gf_coefficients(polynomial)
assert encoding.from_gf_coefficients(coefficients) == polynomial
layout = FieldPolynomialLayout(2, 2)
wires = (4, 0, 5, 2, 3, 1)
words = layout.split(wires)
assert layout.join(words) == wires
assert layout.unpack_basis(layout.pack_basis((1, 2, 3))) == (1, 2, 3)
state = torch.arange(1, 65, dtype=torch.float64) + 1j * torch.arange(64, dtype=torch.float64)
state /= state.norm()
circuit = fq.Circuit(6, inputs=state, dtype=torch.complex128).append(FieldOperation(6, ()))
torch.testing.assert_close(circuit.state().flatten(), state, rtol=0, atol=1e-12)
print("Actual emitted primitives", {})
print("Declared view model", {"Clifford": 0, "T": 0, "measurement": 0, "logical_width": 6})
print("Restored words", words)
